In [ ]:
from collections.abc import Callable
from itertools import count
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import torch
from torch import nn
from torch.optim.adamw import AdamW

In [ ]:
COUNTRY_CODE = "UKR"

In [ ]:
name_df = pd.read_csv(f"../../../data/name_parser/tmp/{COUNTRY_CODE}.csv")

In [ ]:
vectorizer = CountVectorizer(analyzer="char", lowercase=False)
vectorizer.fit_transform(name_df["name"])
unknown = "_"
alphabet = tuple(unknown) + tuple(vectorizer.get_feature_names_out())
alphabet_len = len(alphabet)

In [ ]:
all_categories = "given_name", "middle_name", "family_name"
categories_len = len(all_categories)
# TODO(Sergey Misuk): use fixed value
name_max_len = name_df["name"].str.len().max()

cat_map = dict(zip(all_categories, count()))


def get_target(label: str) -> int:
    return cat_map[label]


cat_map

In [ ]:
def letter_to_index(letter: str) -> int:
    return alphabet.index(letter) if letter in alphabet else alphabet.index(unknown)


# TODO(Sergey Misuk): find out what is that
oob = alphabet_len + 1


def line_to_tensor(line: str) -> torch.Tensor:
    tensor = torch.ones(name_max_len, dtype=torch.long) * oob
    for li, letter in enumerate(line):
        tensor[li] = letter_to_index(letter)
    return tensor

In [ ]:
import torch
from torch.utils.data import Dataset


class CustomDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform: Callable | None = None) -> None:
        self.df = df
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        name = self.df.iloc[idx, self.df.columns.get_loc("name")]
        if self.transform:
            name = self.transform(name)
        label = self.df.iloc[idx, name_df.columns.get_loc("type")]
        label = get_target(label)
        target = torch.tensor(label, dtype=torch.int64)
        return name, target

In [ ]:
dataset = CustomDataset(name_df, line_to_tensor)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
train_dataset, test_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [0.75, 0.10, 0.15], generator=torch.Generator(device=device).manual_seed(2024)
)

In [ ]:
for i in range(3):
    name, label = train_dataset[i]

In [ ]:
from torch.utils.data import DataLoader

batch_size = 128

train_dataloader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=os.cpu_count()
)
test_dataloader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=os.cpu_count()
)
val_dataloader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=os.cpu_count()
)

In [ ]:
torch.manual_seed(42)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


class LSTM(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, output_size: int, num_layers: int = 1) -> None:
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # The nn.Embedding layer returns a new tensor with dimension (sequence_length, 1, hidden_size)
        self.embedding = nn.Embedding(input_size, hidden_size)
        # LSTM layer expects a tensor of dimension (batch_size, sequence_length, hidden_size).
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input_: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(input_.type(torch.IntTensor).to(input_.device))
        h0 = torch.zeros(self.num_layers, embedded.size(0), self.hidden_size).to(input_.device)
        c0 = torch.zeros(self.num_layers, embedded.size(0), self.hidden_size).to(input_.device)
        out, _ = self.lstm(embedded, (h0, c0))
        out = out[:, -1, :]  # get the output of the last time step
        out = self.fc(out)
        return self.softmax(out)


n_hidden = 256

# TODO(Sergey Misuk): find out what is this
vocabulary_size = alphabet_len + 1 + 1  # vocab + oob + 1

rnn = LSTM(vocabulary_size, n_hidden, categories_len, num_layers=2)
rnn.to(device)

In [ ]:
class EarlyStopper:
    def __init__(self, patience: int = 1, min_delta: float = 0) -> None:
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = np.inf

    def early_stop(self, validation_loss: float) -> bool:
        if validation_loss < self.min_validation_loss:
            self.min_validation_loss = validation_loss
            self.counter = 0
        elif validation_loss > (self.min_validation_loss + self.min_delta):
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

In [ ]:
epochs = 100
lr = 0.005

criterion = nn.NLLLoss()
optimizer = AdamW(rnn.parameters(), lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=60, eta_min=1e-5)


early_stopper = EarlyStopper(patience=10)


x = []
training_losses = []
validation_losses = []

valid_mean_min = np.inf

for epoch in range(epochs):
    x.append(epoch)
    # Train
    rnn.train()
    total_loss = torch.Tensor([0.0]).to(device)
    for batch in train_dataloader:
        rnn.zero_grad()
        input_ = batch[0].to(device)
        label = batch[1].to(device)
        output = rnn(input_)
        loss = criterion(output, label)
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            total_loss += loss.item()

    scheduler.step()

    mean = total_loss / len(train_dataloader)
    training_losses.append(mean.cpu())

    rnn.eval()
    validation_loss = torch.Tensor([0.0]).to(device)
    with torch.no_grad():
        for batch in val_dataloader:
            input_ = batch[0].to(device)
            label = batch[1].to(device)
            output = rnn(input_)
            loss = criterion(output, label)
            validation_loss += loss.item()

    val_mean = validation_loss / len(val_dataloader)
    validation_losses.append(val_mean.cpu())
    valid_mean_min = min(valid_mean_min, val_mean.item())

    if early_stopper.early_stop(val_mean.item()):
        break

In [ ]:
torch.save(rnn.state_dict(), f"../../../data/name_parser/models/{COUNTRY_CODE}.pt")
with Path(f"../../../data/name_parser/models/{COUNTRY_CODE}.txt").open("w") as f:
    f.write("".join(alphabet) + "\n")
    f.write(f"{name_max_len}\n")
    f.write(f"{vocabulary_size} {n_hidden} {categories_len}\n")

In [ ]:
def name_parser(name: str) -> str:
    name_tokens = [line_to_tensor(i) for i in name.split()]
    out = [rnn(i.unsqueeze(0).to(device)) for i in name_tokens]
    probs = [torch.exp(i) for i in out]
    out = [torch.argmax(i) for i in probs]
    name_types = [all_categories[i.item()] for i in out]
    return ", ".join(name_types)

In [ ]:
train_on_gpu = torch.cuda.is_available()

if not train_on_gpu:
    pass
else:
    pass

In [ ]:
criterion = nn.NLLLoss()
rnn = LSTM(vocabulary_size, n_hidden, categories_len, num_layers=2)
rnn.load_state_dict(torch.load(f"../../../data/name_parser/models/{COUNTRY_CODE}.pt"))
rnn.to(device)

In [ ]:
# track test loss
test_loss = 0.0


class_correct = [0.0 for i in range(categories_len)]
class_total = [0.0 for i in range(categories_len)]


actual = []
predictions = []

rnn.eval()

for batch in test_dataloader:
    # move tensors to GPU if CUDA is available
    input_ = batch[0].to(device)
    label = batch[1].to(device)
    output = rnn(input_)
    loss = criterion(output, label)
    test_loss += loss.item()
    pred = torch.argmax(output, dim=1)
    correct_tensor = pred.eq(label.data.view_as(pred))
    correct = np.squeeze(correct_tensor.numpy()) if not train_on_gpu else np.squeeze(correct_tensor.cpu().numpy())
    # calculate test accuracy for each object class
    for i in range(label.shape[0]):
        lbl = label.data[i]
        class_correct[lbl.long()] += correct[i].item()
        class_total[lbl.long()] += 1
        # for confusion matrix
        actual.append(all_categories[label.data[i].item()])
        predictions.append(all_categories[pred.data[i].item()])

In [ ]:
name_parser("Володимир Олександрович Зеленський")

In [ ]:
name_parser("Зеленський Володимир Олександрович")

In [ ]:
name_parser("Зеленський Володимир")